In [1]:
import os, cv2, numpy as np, random
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SEED = 15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [4]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN3_7k/train"
IMG_SIZE = (128, 128)

In [5]:
def load_spatial(real_dir, fake_dir, img_size=(128, 128), limit=None):
    X, y = [], []
    def proc(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size)
        img = img.astype(np.float32)/255.0
        return img[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals:
        X.append(proc(os.path.join(real_dir, fn))); y.append(0)
    for fn in fakes:
        X.append(proc(os.path.join(fake_dir, fn))); y.append(1)
    return np.array(X), np.array(y)


In [6]:
X, y = load_spatial(REAL_DIR, FAKE_DIR)
print("Spatial shape:", X.shape, "labels:", y.shape)

Spatial shape: (10575, 128, 128, 1) labels: (10575,)


In [7]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (4,4), activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out)

In [8]:
model = build_branch((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [9]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=20, batch_size=35, callbacks=[early_stop], verbose=1)
print("Test eval:", model.evaluate(Xte, yte, verbose=0))

Epoch 1/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 124s 470ms/step - accuracy: 0.7830 - loss: 0.4104 - val_accuracy: 0.9522 - val_loss: 0.1318
Epoch 2/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 135s 448ms/step - accuracy: 0.9593 - loss: 0.1236 - val_accuracy: 0.9674 - val_loss: 0.0924
Epoch 3/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 146s 466ms/step - accuracy: 0.9706 - loss: 0.0844 - val_accuracy: 0.9768 - val_loss: 0.0796
Epoch 4/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 142s 466ms/step - accuracy: 0.9745 - loss: 0.0795 - val_accuracy: 0.9697 - val_loss: 0.0913
Epoch 5/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 153s 510ms/step - accuracy: 0.9816 - loss: 0.0557 - val_accuracy: 0.9740 - val_loss: 0.0808
Epoch 6/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 152s 551ms/step - accuracy: 0.9874 - loss: 0.0419 - val_accuracy: 0.9697 - val_loss: 0.1007
Epoch 7/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 116s 446ms/step - accuracy: 0.9887 - loss: 0.0354 - val_accuracy: 0.9773 - val_loss: 0.0701
Epoch 8/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 142s 446ms/step - accuracy: 0.9926 -

In [10]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/test"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/test"

X_test,  y_test = load_spatial(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", X_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.4).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))



cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot(cmap="Blues", values_format='d')
plt.title("Confusion Matrix")
plt.show()


misclassified_idx = np.where(y_pred != y_test)[0]
print("Number of misclassified images:", len(misclassified_idx))

n_show = 50
row=5
cols=10

plt.figure(figsize=(20, 10))
for i, idx in enumerate(misclassified_idx[250:300]):
    plt.subplot(row, cols, i+1)
    plt.imshow(X_test[idx])
    plt.title(f"True:{y_test[idx]} Pred:{y_pred[idx]}",fontsize=16)
    plt.axis("off")
plt.suptitle("Misclassified Images")
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.